# Count of Gates for Shift and Phase Oracle in $L \times L$ grid

$NOTE : $ Here we are performing the experiment of Google Colab, if you are performing on another platform or in your local device the gate counts may change.

In [3]:
#!pip install qiskit-aer
#!pip install qiskit

In [4]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit import transpile
#from qiskit.visualization import *
from qiskit.circuit.library import QFT, UnitaryGate
from qiskit.quantum_info import Statevector
from numpy import pi
import numpy as np
from matplotlib import pyplot as plt

# Aer is now a separate package (qiskit-aer)
from qiskit_aer import AerSimulator

In [60]:
Q = 8

In [61]:

N = Q*Q # Total Number of vertex in the grid
l = 4/N # Valule for self loop

In [62]:
# Compute coefficients of the coin
a = 2 / (4 + l)
b = 2 * np.sqrt(l) / (4 + l)
c = 2 * l / (4 + l)

In [63]:
C5 = np.array([
    [a - 1,  a,      a,      a,      b],
    [a,      a - 1,  a,      a,      b],
    [a,      a,      a - 1,  a,      b],
    [a,      a,      a,      a - 1,  b],
    [b,      b,      b,      b,      c - 1]
], dtype=complex)

# Embed into an 8x8 matrix for 3 qubits
C8 = np.zeros((8, 8), dtype=complex)
C8[:5, :5] = C5  # top-left block
for i in range(5, 8):
    C8[i, i] = 1

# Creating the gate for Coin
coin_gate = UnitaryGate(C8, label="Coin")

In [64]:
from qiskit.circuit.library import StatePreparation

coin_init = np.zeros(8, dtype=complex)
coin_init[0] = 1 / np.sqrt(4 + l)           # 000
coin_init[1] = 1 / np.sqrt(4 + l)           # 001
coin_init[2] = 1 / np.sqrt(4 + l)           # 010
coin_init[3] = 1 / np.sqrt(4 + l)           # 011
coin_init[4] = np.sqrt(l) / np.sqrt(4 + l)  # 100

coin_prep = StatePreparation(coin_init)

In [65]:
coin = QuantumRegister(3,'coin')
nodeX = QuantumRegister(int(np.log2(Q)),'vertex_X')
nodeY = QuantumRegister(int(np.log2(Q)),'vertex_y')
classR = ClassicalRegister(int(2*np.log2(Q)),'measure')

one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')

In [66]:
#phase oracle
A = 2 * int(np.log2(Q))
phase_circuit =  QuantumCircuit(A, name=' phase oracle ')
# Mark 100000 for any code
cont = []
for i in range(0,A-1):
    phase_circuit.x(i)
    cont.append(i)
phase_circuit.h(A-1)
phase_circuit.mcx(cont,A-1)
phase_circuit.h(A-1)
for i in range(0,A-1):
    phase_circuit.x(i)
    cont.append(i)
phase_circuit.draw()

┌───┐     ┌───┐
q_0: ┤ X ├──■──┤ X ├
     ├───┤  │  ├───┤
q_1: ┤ X ├──■──┤ X ├
     ├───┤  │  ├───┤
q_2: ┤ X ├──■──┤ X ├
     ├───┤  │  ├───┤
q_3: ┤ X ├──■──┤ X ├
     ├───┤  │  ├───┤
q_4: ┤ X ├──■──┤ X ├
     ├───┤┌─┴─┐├───┤
q_5: ┤ H ├┤ X ├┤ H ├
     └───┘└───┘└───┘

In [67]:
def superposition(circuit, Q):
    num_states = int(2*np.log2(Q))
    for i in range(0,num_states):
        circuit.h(i)

In [68]:
one_step.append(coin_prep, coin)

In [69]:
def shift(circuit, Q):
    num_states = 3 + int(2*np.log2(Q))
    circuit.x(num_states-3)
    circuit.x(num_states-2)
    circuit.x(num_states-1)
    E = int(np.log2(Q))
    D = E
    for i in range(int(np.log2(Q))):
        x = list(range(0,E-1))
        x.append(num_states-1)
        x.append(num_states-2)
        x.append(num_states-3)
        circuit.mcx(x,E-1)
        E = E-1
    circuit.x(num_states-3)
    for i in range(int(np.log2(Q))):
        x = list(range(0,E))
        x.append(num_states-1)
        x.append(num_states-2)
        x.append(num_states-3)
        circuit.mcx(x,E)
        E = E+1
    circuit.x(num_states-3)
    circuit.x(num_states-2)
    E = 2*int(np.log2(Q))
    for i in range(int(np.log2(Q))):
        x = list(range(D,E-1))
        x.append(num_states-1)
        x.append(num_states-2)
        x.append(num_states-3)
        circuit.mcx(x,E-1)
        E = E-1
    circuit.x(num_states-3)
    for i in range(int(np.log2(Q))):
        x = list(range(D,E))
        x.append(num_states-1)
        x.append(num_states-2)
        x.append(num_states-3)
        circuit.mcx(x,E)
        E = E+1
    circuit.x(num_states-1)
    circuit.x(num_states-1)
    circuit.mcx([num_states-1],num_states-3)
    circuit.x(num_states-1)

# 8x8

# Initialization

In [70]:
x = 17 #number of steps
Q = 8
l = 4/(Q*Q)

coin_init = np.zeros(8, dtype=complex)
coin_init[0] = 1 / np.sqrt(4 + l)           # 000
coin_init[1] = 1 / np.sqrt(4 + l)           # 001
coin_init[2] = 1 / np.sqrt(4 + l)           # 010
coin_init[3] = 1 / np.sqrt(4 + l)           # 011
coin_init[4] = np.sqrt(l) / np.sqrt(4 + l)  # 100

coin_prep = StatePreparation(coin_init)
one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
one_step.append(coin_prep, coin)
one = transpile(one_step, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0, seed_transpiler = 42)
one.count_ops()

OrderedDict([('h', 154),
             ('t', 148),
             ('s', 78),
             ('sdg', 26),
             ('z', 6),
             ('cx', 2)])

# For one Coin

In [71]:
one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
one_step.append(coin_gate, coin)
#one_step.decompose().decompose().decompose().count_ops()
one = transpile(one_step, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0, seed_transpiler = 42)
one.count_ops()

OrderedDict([('h', 9122),
             ('t', 9025),
             ('s', 4581),
             ('sdg', 142),
             ('x', 36),
             ('z', 19),
             ('cx', 19),
             ('tdg', 6)])

# Phase Oracle

In [72]:
#phase oracle
A = 2 * int(np.log2(Q))
phase_circuit =  QuantumCircuit(A, name=' phase oracle ')
# Mark 100000 for any code
cont = []
for i in range(0,A-1):
    phase_circuit.x(i)
    cont.append(i)
phase_circuit.h(A-1)
phase_circuit.mcx(cont,A-1)
phase_circuit.h(A-1)
for i in range(0,A-1):
    phase_circuit.x(i)
    cont.append(i)
phase_circuit.decompose().decompose().count_ops()
one = transpile(phase_circuit, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0, seed_transpiler=42)
one.count_ops()

OrderedDict([('h', 5754),
             ('t', 5719),
             ('s', 3036),
             ('cx', 84),
             ('x', 28),
             ('tdg', 26),
             ('sdg', 21)])

# For one shift

In [73]:
Q = 8
one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
shift(one_step,Q)

In [74]:
one = transpile(one_step, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0, seed_transpiler=42)

In [75]:
one.count_ops()

OrderedDict([('cx', 289), ('t', 192), ('tdg', 174), ('h', 156), ('x', 46)])

# One  Complete Step

In [76]:
one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
superposition(one_step,Q)
one_step.append(coin_prep,coin)
one_step.append(coin_gate,coin)
one_step.append(phase_circuit,[0,1,2,3,4,5])
shift(one_step,Q)


In [77]:
one = transpile(one_step, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0, seed_transpiler=42)
one.count_ops()

OrderedDict([('h', 9490),
             ('t', 9429),
             ('s', 4659),
             ('cx', 406),
             ('tdg', 237),
             ('sdg', 168),
             ('x', 108),
             ('z', 25)])

# For Complete Walk

In [78]:
one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
superposition(one_step,Q)
one_step.append(coin_prep,coin)
for i in range(17-2):
    one_step.append(coin_gate,coin)
    one_step.append(phase_circuit,[0,1,2,3,4,5])
    shift(one_step,Q)

In [79]:
one = transpile(one_step, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0, seed_transpiler=42)
one.count_ops()

OrderedDict([('h', 140250),
             ('t', 139531),
             ('s', 68793),
             ('cx', 6314),
             ('tdg', 3709),
             ('sdg', 2156),
             ('x', 1676),
             ('z', 291)])

In [80]:
import json
from qiskit import QuantumCircuit, transpile

def ft_resource_metrics(qc: QuantumCircuit):
    n = qc.num_qubits
    qindex = {q: i for i, q in enumerate(qc.qubits)}
    depth = [0] * n
    tdepth = [0] * n
    cxdepth = [0] * n
    logical_depth = 0
    max_tdepth = 0
    max_cxdepth = 0
    t_count = 0
    cx_count = 0
    measure_count = 0

    for inst in qc.data:
        name = inst.operation.name
        qs = inst.qubits
        nq = len(qs)
        if nq == 0: continue
        if nq == 1:
            q0 = qindex[qs[0]]
            # Full depth
            d = depth[q0] + 1
            depth[q0] = d
            if d > logical_depth: logical_depth = d
            # T depth
            td = tdepth[q0]
            if name in {"t", "tdg"}:
                t_count += 1
                td += 1
                if td > max_tdepth: max_tdepth = td
            tdepth[q0] = td
            if name == "measure": measure_count += 1
        elif nq == 2:
            q0, q1 = qindex[qs[0]], qindex[qs[1]]
            # Full depth
            d0, d1 = depth[q0], depth[q1]
            d = (d0 if d0 > d1 else d1) + 1
            depth[q0] = depth[q1] = d
            if d > logical_depth: logical_depth = d
            # T depth
            td0, td1 = tdepth[q0], tdepth[q1]
            td = td0 if td0 > td1 else td1
            if name in {"t", "tdg"}:
                t_count += 1
                td += 1
                if td > max_tdepth: max_tdepth = td
            tdepth[q0] = tdepth[q1] = td
            # CX depth
            cd0, cd1 = cxdepth[q0], cxdepth[q1]
            cd = cd0 if cd0 > cd1 else cd1
            if name == "cx":
                cx_count += 1
                cd += 1
                if cd > max_cxdepth: max_cxdepth = cd
            cxdepth[q0] = cxdepth[q1] = cd
        else:
            inds = [qindex[q] for q in qs]
            # Full depth
            d = max(depth[q] for q in inds) + 1
            for q in inds: depth[q] = d
            if d > logical_depth: logical_depth = d
            # T depth
            td = max(tdepth[q] for q in inds)
            if name in {"t", "tdg"}:
                t_count += 1
                td += 1
                if td > max_tdepth: max_tdepth = td
            for q in inds: tdepth[q] = td
            # CX depth
            cd = max(cxdepth[q] for q in inds)
            if name == "cx":
                cx_count += 1
                cd += 1
                if cd > max_cxdepth: max_cxdepth = cd
            for q in inds: cxdepth[q] = cd
    return {
        "logical_qubits": n,
        "logical_depth": logical_depth,
        "t_count": t_count,
        "t_depth": max_tdepth,         # <-- THIS IS YOUR NON-CLIFFORD DEPTH
        "cx_count": cx_count,
        "cx_depth": max_cxdepth,
        "measurements": measure_count,
    }
# ==========================================
# 2. EXECUTION & PRINTING BLOCK
# ==========================================

# I highly recommend changing optimization_level to 2 or 3 to
# cancel out redundant gates and give a more realistic metric!
one = transpile(
    one_step,
    basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],
    optimization_level=0,
    seed_transpiler=42
)
print("--- Standard Metrics ---")
print("Total circuit depth (Qiskit):", one.depth())
print("\n--- Fault-Tolerant Metrics ---")
metrics = ft_resource_metrics(one)
print(json.dumps(metrics, indent=4))

--- Standard Metrics ---
Total circuit depth (Qiskit): 213156

--- Fault-Tolerant Metrics ---
{
    "logical_qubits": 9,
    "logical_depth": 213156,
    "t_count": 143240,
    "t_depth": 83925,
    "cx_count": 6314,
    "cx_depth": 5525,
    "measurements": 0
}


# 16x16

In [23]:
Q = 16
N = Q*Q # Total Number of vertex in the grid
l = 4/N
# Compute coefficients of the coin
a = 2 / (4 + l)
b = 2 * np.sqrt(l) / (4 + l)
c = 2 * l / (4 + l)
C5 = np.array([
    [a - 1,  a,      a,      a,      b],
    [a,      a - 1,  a,      a,      b],
    [a,      a,      a - 1,  a,      b],
    [a,      a,      a,      a - 1,  b],
    [b,      b,      b,      b,      c - 1]
], dtype=complex)

# Embed into an 8x8 matrix for 3 qubits
C8 = np.zeros((8, 8), dtype=complex)
C8[:5, :5] = C5  # top-left block
for i in range(5, 8):
    C8[i, i] = 1

# Creating the gate for Coin
coin_gate = UnitaryGate(C8, label="Coin")
from qiskit.circuit.library import StatePreparation

coin_init = np.zeros(8, dtype=complex)
coin_init[0] = 1 / np.sqrt(4 + l)           # 000
coin_init[1] = 1 / np.sqrt(4 + l)           # 001
coin_init[2] = 1 / np.sqrt(4 + l)           # 010
coin_init[3] = 1 / np.sqrt(4 + l)           # 011
coin_init[4] = np.sqrt(l) / np.sqrt(4 + l)  # 100

coin_prep = StatePreparation(coin_init)
coin = QuantumRegister(3,'coin')
nodeX = QuantumRegister(int(np.log2(Q)),'vertex_X')
nodeY = QuantumRegister(int(np.log2(Q)),'vertex_y')
classR = ClassicalRegister(int(2*np.log2(Q)),'measure')


one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
one_step.append(coin_prep, coin)
one = transpile(one_step, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0, seed_transpiler=42)
one.count_ops()


OrderedDict([('h', 157),
             ('t', 150),
             ('s', 68),
             ('sdg', 26),
             ('z', 6),
             ('cx', 2),
             ('x', 1)])

# Phase Oracle

In [24]:
#phase oracle
A = 2 * int(np.log2(Q))
phase_circuit =  QuantumCircuit(A, name=' phase oracle ')
# Mark 100000 for any code
cont = []
for i in range(0,A-1):
    phase_circuit.x(i)
    cont.append(i)
phase_circuit.h(A-1)
phase_circuit.mcx(cont,A-1)
phase_circuit.h(A-1)
for i in range(0,A-1):
    phase_circuit.x(i)
    cont.append(i)
# phase_circuit.draw()
phase_circuit.decompose().count_ops()
one = transpile(phase_circuit, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0, seed_transpiler=42)
one.count_ops()

OrderedDict([('h', 17749),
             ('t', 17616),
             ('s', 9433),
             ('cx', 192),
             ('x', 98),
             ('sdg', 68),
             ('tdg', 34)])

# For one Shift Oporation

In [25]:
one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
shift(one_step,Q)
one_step.decompose().count_ops()
one = transpile(one_step, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0, seed_transpiler=42)
one.count_ops()

OrderedDict([('cx', 457), ('t', 304), ('tdg', 280), ('h', 256), ('x', 70)])

# Coin

In [26]:
one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
one_step.append(coin_gate, coin)
#one_step.decompose().decompose().decompose().count_ops()
one = transpile(one_step, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0, seed_transpiler=42)
one.count_ops()

OrderedDict([('h', 9140),
             ('t', 9034),
             ('s', 4653),
             ('sdg', 135),
             ('x', 33),
             ('z', 27),
             ('cx', 19),
             ('tdg', 6)])

# One Step of Walk

In [27]:
one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
superposition(one_step,Q)
one_step.append(coin_prep,coin)
one_step.append(coin_gate,coin)
one_step.append(phase_circuit,[0,1,2,3,4,5,6,7])
shift(one_step,Q)
one = transpile(one_step, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0, seed_transpiler=42)
one.count_ops()

OrderedDict([('h', 9649),
             ('t', 9592),
             ('s', 4721),
             ('cx', 634),
             ('tdg', 381),
             ('sdg', 161),
             ('x', 144),
             ('z', 33)])

# Complete Walk

In [28]:
one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
superposition(one_step,Q)
one_step.append(coin_prep,coin)
for i in range(33-2):
    one_step.append(coin_gate,coin)
    one_step.append(phase_circuit,[0,1,2,3,4,5,6,7])
    shift(one_step,Q)

one = transpile(one_step, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0, seed_transpiler=42)
one.count_ops()

OrderedDict([('h', 294709),
             ('t', 293452),
             ('s', 144311),
             ('cx', 20494),
             ('tdg', 12381),
             ('x', 4614),
             ('sdg', 4211),
             ('z', 843)])

In [29]:
import json
from qiskit import QuantumCircuit, transpile

def ft_resource_metrics(qc: QuantumCircuit):
    n = qc.num_qubits
    qindex = {q: i for i, q in enumerate(qc.qubits)}
    depth = [0] * n
    tdepth = [0] * n
    cxdepth = [0] * n
    logical_depth = 0
    max_tdepth = 0
    max_cxdepth = 0
    t_count = 0
    cx_count = 0
    measure_count = 0

    for inst in qc.data:
        name = inst.operation.name
        qs = inst.qubits
        nq = len(qs)
        if nq == 0: continue
        if nq == 1:
            q0 = qindex[qs[0]]
            # Full depth
            d = depth[q0] + 1
            depth[q0] = d
            if d > logical_depth: logical_depth = d
            # T depth
            td = tdepth[q0]
            if name in {"t", "tdg"}:
                t_count += 1
                td += 1
                if td > max_tdepth: max_tdepth = td
            tdepth[q0] = td
            if name == "measure": measure_count += 1
        elif nq == 2:
            q0, q1 = qindex[qs[0]], qindex[qs[1]]
            # Full depth
            d0, d1 = depth[q0], depth[q1]
            d = (d0 if d0 > d1 else d1) + 1
            depth[q0] = depth[q1] = d
            if d > logical_depth: logical_depth = d
            # T depth
            td0, td1 = tdepth[q0], tdepth[q1]
            td = td0 if td0 > td1 else td1
            if name in {"t", "tdg"}:
                t_count += 1
                td += 1
                if td > max_tdepth: max_tdepth = td
            tdepth[q0] = tdepth[q1] = td
            # CX depth
            cd0, cd1 = cxdepth[q0], cxdepth[q1]
            cd = cd0 if cd0 > cd1 else cd1
            if name == "cx":
                cx_count += 1
                cd += 1
                if cd > max_cxdepth: max_cxdepth = cd
            cxdepth[q0] = cxdepth[q1] = cd
        else:
            inds = [qindex[q] for q in qs]
            # Full depth
            d = max(depth[q] for q in inds) + 1
            for q in inds: depth[q] = d
            if d > logical_depth: logical_depth = d
            # T depth
            td = max(tdepth[q] for q in inds)
            if name in {"t", "tdg"}:
                t_count += 1
                td += 1
                if td > max_tdepth: max_tdepth = td
            for q in inds: tdepth[q] = td
            # CX depth
            cd = max(cxdepth[q] for q in inds)
            if name == "cx":
                cx_count += 1
                cd += 1
                if cd > max_cxdepth: max_cxdepth = cd
            for q in inds: cxdepth[q] = cd
    return {
        "logical_qubits": n,
        "logical_depth": logical_depth,
        "t_count": t_count,
        "t_depth": max_tdepth,         # <-- THIS IS YOUR NON-CLIFFORD DEPTH
        "cx_count": cx_count,
        "cx_depth": max_cxdepth,
        "measurements": measure_count,
    }
# ==========================================
# 2. EXECUTION & PRINTING BLOCK
# ==========================================

# I highly recommend changing optimization_level to 2 or 3 to
# cancel out redundant gates and give a more realistic metric!
one = transpile(
    one_step,
    basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],
    optimization_level=0,
    seed_transpiler=42
)
print("--- Standard Metrics ---")
print("Total circuit depth (Qiskit):", one.depth())
print("\n--- Fault-Tolerant Metrics ---")
metrics = ft_resource_metrics(one)
print(json.dumps(metrics, indent=4))

--- Standard Metrics ---
Total circuit depth (Qiskit): 457511

--- Fault-Tolerant Metrics ---
{
    "logical_qubits": 11,
    "logical_depth": 457511,
    "t_count": 305833,
    "t_depth": 179664,
    "cx_count": 20494,
    "cx_depth": 17505,
    "measurements": 0
}


# 32x32

In [30]:
Q = 32
N = Q*Q # Total Number of vertex in the grid
l = 4/N
# Compute coefficients of the coin
a = 2 / (4 + l)
b = 2 * np.sqrt(l) / (4 + l)
c = 2 * l / (4 + l)
C5 = np.array([
    [a - 1,  a,      a,      a,      b],
    [a,      a - 1,  a,      a,      b],
    [a,      a,      a - 1,  a,      b],
    [a,      a,      a,      a - 1,  b],
    [b,      b,      b,      b,      c - 1]
], dtype=complex)

# Embed into an 8x8 matrix for 3 qubits
C8 = np.zeros((8, 8), dtype=complex)
C8[:5, :5] = C5  # top-left block
for i in range(5, 8):
    C8[i, i] = 1

# Creating the gate for Coin
coin_gate = UnitaryGate(C8, label="Coin")
from qiskit.circuit.library import StatePreparation

coin_init = np.zeros(8, dtype=complex)
coin_init[0] = 1 / np.sqrt(4 + l)           # 000
coin_init[1] = 1 / np.sqrt(4 + l)           # 001
coin_init[2] = 1 / np.sqrt(4 + l)           # 010
coin_init[3] = 1 / np.sqrt(4 + l)           # 011
coin_init[4] = np.sqrt(l) / np.sqrt(4 + l)  # 100

coin_prep = StatePreparation(coin_init)
coin = QuantumRegister(3,'coin')
nodeX = QuantumRegister(int(np.log2(Q)),'vertex_X')
nodeY = QuantumRegister(int(np.log2(Q)),'vertex_y')
classR = ClassicalRegister(int(2*np.log2(Q)),'measure')

one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
one_step.append(coin_prep, coin)
one = transpile(one_step, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0, seed_transpiler=42)
one.count_ops()


OrderedDict([('h', 156),
             ('t', 149),
             ('s', 79),
             ('sdg', 26),
             ('z', 6),
             ('cx', 2)])

# phase oracle

In [31]:

A = 2 * int(np.log2(Q))
phase_circuit =  QuantumCircuit(A, name=' phase oracle ')
# Mark 100000 for any code
cont = []
for i in range(0,A-1):
    phase_circuit.x(i)
    cont.append(i)
phase_circuit.h(A-1)
phase_circuit.mcx(cont,A-1)
phase_circuit.h(A-1)
for i in range(0,A-1):
    phase_circuit.x(i)
    cont.append(i)
phase_circuit.decompose().count_ops()
one = transpile(phase_circuit, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0, seed_transpiler=42)
one.count_ops()

OrderedDict([('h', 28959),
             ('t', 28770),
             ('s', 15092),
             ('cx', 344),
             ('x', 151),
             ('sdg', 110),
             ('tdg', 78)])

# For One step Shift

In [32]:
one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
shift(one_step,Q)
one_step.decompose().count_ops()
one = transpile(one_step, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0, seed_transpiler=42)
one.count_ops()

OrderedDict([('cx', 661), ('t', 440), ('tdg', 410), ('h', 380), ('x', 106)])

# Coin

In [33]:

one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
one_step.append(coin_gate, coin)
#one_step.decompose().decompose().decompose().count_ops()
one = transpile(one_step, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0, seed_transpiler=42)
one.count_ops()

OrderedDict([('h', 9113),
             ('t', 9005),
             ('s', 4628),
             ('sdg', 137),
             ('x', 35),
             ('z', 22),
             ('cx', 19),
             ('tdg', 6)])

# One Step of Walk

In [34]:
one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
superposition(one_step,Q)
one_step.append(coin_prep,coin)
one_step.append(coin_gate,coin)
one_step.append(phase_circuit,[0,1,2,3,4,5,6,7,8,9])
shift(one_step,Q)
one = transpile(one_step, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0, seed_transpiler=42)
one.count_ops()

OrderedDict([('h', 9791),
             ('t', 9746),
             ('s', 4707),
             ('cx', 910),
             ('tdg', 557),
             ('x', 201),
             ('sdg', 163),
             ('z', 28)])

# Complete Walk

In [35]:
one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
superposition(one_step,Q)
one_step.append(coin_prep,coin)
for i in range(75-2):
    one_step.append(coin_gate,coin)
    one_step.append(phase_circuit,[0,1,2,3,4,5,6,7,8,9])
    shift(one_step,Q)

one = transpile(one_step, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0, seed_transpiler=42)
one.count_ops()

OrderedDict([('h', 704663),
             ('t', 702746),
             ('s', 337923),
             ('cx', 69310),
             ('tdg', 42605),
             ('x', 15393),
             ('sdg', 10027),
             ('z', 1612)])

In [36]:
import json
from qiskit import QuantumCircuit, transpile

def ft_resource_metrics(qc: QuantumCircuit):
    n = qc.num_qubits
    qindex = {q: i for i, q in enumerate(qc.qubits)}
    depth = [0] * n
    tdepth = [0] * n
    cxdepth = [0] * n
    logical_depth = 0
    max_tdepth = 0
    max_cxdepth = 0
    t_count = 0
    cx_count = 0
    measure_count = 0

    for inst in qc.data:
        name = inst.operation.name
        qs = inst.qubits
        nq = len(qs)
        if nq == 0: continue
        if nq == 1:
            q0 = qindex[qs[0]]
            # Full depth
            d = depth[q0] + 1
            depth[q0] = d
            if d > logical_depth: logical_depth = d
            # T depth
            td = tdepth[q0]
            if name in {"t", "tdg"}:
                t_count += 1
                td += 1
                if td > max_tdepth: max_tdepth = td
            tdepth[q0] = td
            if name == "measure": measure_count += 1
        elif nq == 2:
            q0, q1 = qindex[qs[0]], qindex[qs[1]]
            # Full depth
            d0, d1 = depth[q0], depth[q1]
            d = (d0 if d0 > d1 else d1) + 1
            depth[q0] = depth[q1] = d
            if d > logical_depth: logical_depth = d
            # T depth
            td0, td1 = tdepth[q0], tdepth[q1]
            td = td0 if td0 > td1 else td1
            if name in {"t", "tdg"}:
                t_count += 1
                td += 1
                if td > max_tdepth: max_tdepth = td
            tdepth[q0] = tdepth[q1] = td
            # CX depth
            cd0, cd1 = cxdepth[q0], cxdepth[q1]
            cd = cd0 if cd0 > cd1 else cd1
            if name == "cx":
                cx_count += 1
                cd += 1
                if cd > max_cxdepth: max_cxdepth = cd
            cxdepth[q0] = cxdepth[q1] = cd
        else:
            inds = [qindex[q] for q in qs]
            # Full depth
            d = max(depth[q] for q in inds) + 1
            for q in inds: depth[q] = d
            if d > logical_depth: logical_depth = d
            # T depth
            td = max(tdepth[q] for q in inds)
            if name in {"t", "tdg"}:
                t_count += 1
                td += 1
                if td > max_tdepth: max_tdepth = td
            for q in inds: tdepth[q] = td
            # CX depth
            cd = max(cxdepth[q] for q in inds)
            if name == "cx":
                cx_count += 1
                cd += 1
                if cd > max_cxdepth: max_cxdepth = cd
            for q in inds: cxdepth[q] = cd
    return {
        "logical_qubits": n,
        "logical_depth": logical_depth,
        "t_count": t_count,
        "t_depth": max_tdepth,         # <-- THIS IS YOUR NON-CLIFFORD DEPTH
        "cx_count": cx_count,
        "cx_depth": max_cxdepth,
        "measurements": measure_count,
    }
# ==========================================
# 2. EXECUTION & PRINTING BLOCK
# ==========================================

# I highly recommend changing optimization_level to 2 or 3 to
# cancel out redundant gates and give a more realistic metric!
one = transpile(
    one_step,
    basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],
    optimization_level=0,
    seed_transpiler=42
)
print("--- Standard Metrics ---")
print("Total circuit depth (Qiskit):", one.depth())
print("\n--- Fault-Tolerant Metrics ---")
metrics = ft_resource_metrics(one)
print(json.dumps(metrics, indent=4))

--- Standard Metrics ---
Total circuit depth (Qiskit): 1106607

--- Fault-Tolerant Metrics ---
{
    "logical_qubits": 13,
    "logical_depth": 1106607,
    "t_count": 745351,
    "t_depth": 434749,
    "cx_count": 69310,
    "cx_depth": 55396,
    "measurements": 0
}


# 64x64

In [15]:
Q = 64
N = Q*Q # Total Number of vertex in the grid
l = 4/N
# Compute coefficients of the coin
a = 2 / (4 + l)
b = 2 * np.sqrt(l) / (4 + l)
c = 2 * l / (4 + l)
C5 = np.array([
    [a - 1,  a,      a,      a,      b],
    [a,      a - 1,  a,      a,      b],
    [a,      a,      a - 1,  a,      b],
    [a,      a,      a,      a - 1,  b],
    [b,      b,      b,      b,      c - 1]
], dtype=complex)

# Embed into an 8x8 matrix for 3 qubits
C8 = np.zeros((8, 8), dtype=complex)
C8[:5, :5] = C5  # top-left block
for i in range(5, 8):
    C8[i, i] = 1

# Creating the gate for Coin
coin_gate = UnitaryGate(C8, label="Coin")
from qiskit.circuit.library import StatePreparation

coin_init = np.zeros(8, dtype=complex)
coin_init[0] = 1 / np.sqrt(4 + l)           # 000
coin_init[1] = 1 / np.sqrt(4 + l)           # 001
coin_init[2] = 1 / np.sqrt(4 + l)           # 010
coin_init[3] = 1 / np.sqrt(4 + l)           # 011
coin_init[4] = np.sqrt(l) / np.sqrt(4 + l)  # 100

coin_prep = StatePreparation(coin_init)
coin = QuantumRegister(3,'coin')
nodeX = QuantumRegister(int(np.log2(Q)),'vertex_X')
nodeY = QuantumRegister(int(np.log2(Q)),'vertex_y')
classR = ClassicalRegister(int(2*np.log2(Q)),'measure')

one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
one_step.append(coin_prep, coin)
one = transpile(one_step, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0, seed_transpiler=42)
one.count_ops()


OrderedDict([('h', 151),
             ('t', 144),
             ('s', 76),
             ('sdg', 26),
             ('z', 6),
             ('cx', 2),
             ('x', 1)])

# Phase Oracle

In [16]:
#phase oracle
A = 2 * int(np.log2(Q))
phase_circuit =  QuantumCircuit(A, name=' phase oracle ')
# Mark 100000 for any code
cont = []
for i in range(0,A-1):
    phase_circuit.x(i)
    cont.append(i)
phase_circuit.h(A-1)
phase_circuit.mcx(cont,A-1)
phase_circuit.h(A-1)
for i in range(0,A-1):
    phase_circuit.x(i)
    cont.append(i)
phase_circuit.decompose().count_ops()
one = transpile(phase_circuit, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0, seed_transpiler=42)
one.count_ops()

OrderedDict([('h', 32655),
             ('t', 32490),
             ('s', 17010),
             ('cx', 576),
             ('tdg', 202),
             ('x', 180),
             ('sdg', 124)])

# For one step Shift

In [17]:
one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
shift(one_step,Q)
one_step.decompose().count_ops()
one = transpile(one_step, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0, seed_transpiler=42)
one.count_ops()

OrderedDict([('cx', 901), ('t', 600), ('tdg', 564), ('h', 528), ('x', 154)])

# Coin

In [18]:
one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
one_step.append(coin_gate, coin)
#one_step.decompose().decompose().decompose().count_ops()
one = transpile(one_step, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0, seed_transpiler=42)
one.count_ops()

OrderedDict([('h', 9116),
             ('t', 9018),
             ('s', 4607),
             ('sdg', 141),
             ('x', 37),
             ('z', 24),
             ('cx', 19),
             ('tdg', 6)])

# One Step Walk

In [19]:
one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
superposition(one_step,Q)
one_step.append(coin_prep,coin)
one_step.append(coin_gate,coin)
one_step.append(phase_circuit,[0,1,2,3,4,5,6,7,8,9,10,11])
shift(one_step,Q)
one = transpile(one_step, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0, seed_transpiler=42)
one.count_ops()

OrderedDict([('h', 9991),
             ('t', 9970),
             ('s', 4683),
             ('cx', 1234),
             ('tdg', 765),
             ('x', 276),
             ('sdg', 167),
             ('z', 30)])

# Complete Walk

In [20]:
one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
superposition(one_step,Q)
one_step.append(coin_prep,coin)
for i in range(165-2):
    one_step.append(coin_gate,coin)
    one_step.append(phase_circuit,[0,1,2,3,4,5,6,7,8,9,10,11])
    shift(one_step,Q)

one = transpile(one_step, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0, seed_transpiler=42)
one.count_ops()

OrderedDict([('h', 1607635),
             ('t', 1607614),
             ('s', 751017),
             ('cx', 209566),
             ('tdg', 130365),
             ('x', 47094),
             ('sdg', 23009),
             ('z', 3918)])

In [22]:
import json
from qiskit import QuantumCircuit, transpile

def ft_resource_metrics(qc: QuantumCircuit):
    n = qc.num_qubits
    qindex = {q: i for i, q in enumerate(qc.qubits)}
    depth = [0] * n
    tdepth = [0] * n
    cxdepth = [0] * n
    logical_depth = 0
    max_tdepth = 0
    max_cxdepth = 0
    t_count = 0
    cx_count = 0
    measure_count = 0

    for inst in qc.data:
        name = inst.operation.name
        qs = inst.qubits
        nq = len(qs)
        if nq == 0: continue
        if nq == 1:
            q0 = qindex[qs[0]]
            # Full depth
            d = depth[q0] + 1
            depth[q0] = d
            if d > logical_depth: logical_depth = d
            # T depth
            td = tdepth[q0]
            if name in {"t", "tdg"}:
                t_count += 1
                td += 1
                if td > max_tdepth: max_tdepth = td
            tdepth[q0] = td
            if name == "measure": measure_count += 1
        elif nq == 2:
            q0, q1 = qindex[qs[0]], qindex[qs[1]]
            # Full depth
            d0, d1 = depth[q0], depth[q1]
            d = (d0 if d0 > d1 else d1) + 1
            depth[q0] = depth[q1] = d
            if d > logical_depth: logical_depth = d
            # T depth
            td0, td1 = tdepth[q0], tdepth[q1]
            td = td0 if td0 > td1 else td1
            if name in {"t", "tdg"}:
                t_count += 1
                td += 1
                if td > max_tdepth: max_tdepth = td
            tdepth[q0] = tdepth[q1] = td
            # CX depth
            cd0, cd1 = cxdepth[q0], cxdepth[q1]
            cd = cd0 if cd0 > cd1 else cd1
            if name == "cx":
                cx_count += 1
                cd += 1
                if cd > max_cxdepth: max_cxdepth = cd
            cxdepth[q0] = cxdepth[q1] = cd
        else:
            inds = [qindex[q] for q in qs]
            # Full depth
            d = max(depth[q] for q in inds) + 1
            for q in inds: depth[q] = d
            if d > logical_depth: logical_depth = d
            # T depth
            td = max(tdepth[q] for q in inds)
            if name in {"t", "tdg"}:
                t_count += 1
                td += 1
                if td > max_tdepth: max_tdepth = td
            for q in inds: tdepth[q] = td
            # CX depth
            cd = max(cxdepth[q] for q in inds)
            if name == "cx":
                cx_count += 1
                cd += 1
                if cd > max_cxdepth: max_cxdepth = cd
            for q in inds: cxdepth[q] = cd
    return {
        "logical_qubits": n,
        "logical_depth": logical_depth,
        "t_count": t_count,
        "t_depth": max_tdepth,         # <-- THIS IS YOUR NON-CLIFFORD DEPTH
        "cx_count": cx_count,
        "cx_depth": max_cxdepth,
        "measurements": measure_count,
    }
# ==========================================
# 2. EXECUTION & PRINTING BLOCK
# ==========================================

# I highly recommend changing optimization_level to 2 or 3 to
# cancel out redundant gates and give a more realistic metric!
one = transpile(
    one_step,
    basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],
    optimization_level=0,
    seed_transpiler=42
)
print("--- Standard Metrics ---")
print("Total circuit depth (Qiskit):", one.depth())
print("\n--- Fault-Tolerant Metrics ---")
metrics = ft_resource_metrics(one)
print(json.dumps(metrics, indent=4))

--- Standard Metrics ---
Total circuit depth (Qiskit): 2555755

--- Fault-Tolerant Metrics ---
{
    "logical_qubits": 15,
    "logical_depth": 2555755,
    "t_count": 1737979,
    "t_depth": 1007102,
    "cx_count": 209566,
    "cx_depth": 158579,
    "measurements": 0
}


# VERSION

In [ ]:
import qiskit
import qiskit_aer
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit import transpile
#from qiskit.visualization import *
from qiskit.circuit.library import QFT, UnitaryGate
from qiskit.quantum_info import Statevector
from numpy import pi
import numpy as np
from matplotlib import pyplot as plt
from qiskit_aer import AerSimulator

# ==========================================
# 1. Define Circuit & Transpilation Settings
# ==========================================
opt_level = 0
transpile_seed = 42

# Decomposition methods for MCX/MCZ in Qiskit are set during circuit building.
# Common modes: 'noancilla', 'recursion', 'v-chain', 'v-chain-dirty'
mcx_decomp_mode = 'noancilla'
ancillas_permitted = "None" # Would be "Clean" for 'v-chain' or "Dirty" for 'v-chain-dirty'

# Create a sample circuit to demonstrate
qc = QuantumCircuit(4)
qc.h(0)
# Multi-controlled X gate using the specified decomposition mode
qc.mcx(control_qubits=[0, 1, 2], target_qubit=3, mode=mcx_decomp_mode)
qc.measure_all()

# ==========================================
# 2. Setup Backend & Transpile
# ==========================================
backend = AerSimulator()

# Extract hardware/simulator constraints
basis_gates = backend.configuration().basis_gates
coupling_map = backend.configuration().coupling_map

# Run the transpiler
transpiled_qc = transpile(
    qc,
    backend=backend,
    basis_gates=basis_gates,
    optimization_level=opt_level,
    seed_transpiler=transpile_seed
)

# ==========================================
# 3. Print the Requested Metadata
# ==========================================
print("--- Versions ---")
print(f"Qiskit version:               {qiskit.__version__}")
print(f"Simulator/Compiler version:   {qiskit_aer.__version__} (qiskit-aer)")

print("\n--- Transpiler & Hardware Configuration ---")
print(f"Basis gate set:               {basis_gates}")
print(f"Coupling map, if any:         {coupling_map}")
print(f"Transpiler opt level:         {opt_level}")
print(f"Seed for transpilation:       {transpile_seed}")

print("\n--- Circuit Specifics ---")
print(f"Multi-controlled X/Z decomp:  {mcx_decomp_mode} (set via `mode` arg in circuit.mcx)")
print(f"Clean/dirty ancillas allowed: {ancillas_permitted}")
print(f"Global phase:                 {transpiled_qc.global_phase} (Tracked by Qiskit, usually ignored by physical hardware unless doing phase estimation)")

print("\n--- Metrics & Reporting Context ---")
# Depth
print(f"Reported depth value:         {transpiled_qc.depth()}")
print("Is depth logical or hardware? LOGICAL DEPENDENCY DEPTH.")
print("                              (Hardware scheduled depth requires running a Scheduling pass like ASAP/ALAP.)")

# Counts context
print("Counts status:                AFTER hardware routing.")
print("                              (Because execution happens on the `transpiled_qc` which has already been routed to the backend's coupling map.)")

--- Versions ---
Qiskit version:               2.5.2
Simulator/Compiler version:   0.17.2 (qiskit-aer)

--- Transpiler & Hardware Configuration ---
Basis gate set:               ['ccx', 'ccz', 'cp', 'crx', 'cry', 'crz', 'cswap', 'csx', 'cu', 'cu1', 'cu2', 'cu3', 'cx', 'cy', 'cz', 'diagonal', 'ecr', 'h', 'id', 'mcp', 'mcphase', 'mcr', 'mcrx', 'mcry', 'mcrz', 'mcswap', 'mcsx', 'mcu', 'mcu1', 'mcu2', 'mcu3', 'mcx', 'mcx_gray', 'mcy', 'mcz', 'multiplexer', 'p', 'pauli', 'r', 'roerror', 'rx', 'rxx', 'ry', 'ryy', 'rz', 'rzx', 'rzz', 's', 'sdg', 'store', 'swap', 'sx', 'sxdg', 't', 'tdg', 'u', 'u1', 'u2', 'u3', 'unitary', 'x', 'y', 'z', 'break_loop', 'continue_loop', 'delay', 'for_loop', 'if_else', 'initialize', 'kraus', 'qerror_loc', 'quantum_channel', 'reset', 'roerror', 'save_amplitudes', 'save_amplitudes_sq', 'save_clifford', 'save_density_matrix', 'save_expval', 'save_expval_var', 'save_matrix_product_state', 'save_probabilities', 'save_probabilities_dict', 'save_stabilizer', 'save_state'

/tmp/ipykernel_829/713467646.py:28: DeprecationWarning: ``qiskit.circuit.quantumcircuit.QuantumCircuit.mcx()``'s argument ``mode`` is deprecated as of Qiskit 2.1. It will be removed no earlier than 3 months after the release date. Instead, add a generic MCXGate to the circuit and specify the synthesis method via the ``hls_config`` in the transpilation. Alternatively, specific decompositions are available at https://qisk.it/mcx.
  qc.mcx(control_qubits=[0, 1, 2], target_qubit=3, mode=mcx_decomp_mode)
/usr/local/lib/python3.12/dist-packages/qiskit/transpiler/preset_passmanagers/generate_preset_pass_manager.py:242: UserWarning: Providing `coupling_map` and/or `basis_gates` along with `backend` is not recommended, as this will invalidate the backend's gate durations and error rates.
  common_options = _parse_common_options(


In [ ]:
!python --version

Python 3.12.13
